In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, upper, avg, round

# 1. Initialize a SparkSession
spark = SparkSession.builder \
    .appName("PySparkTransformationsDemo") \
    .getOrCreate()

# 2. Create a sample dataset
# Represents employees with columns: ID, Name, Department, Salary, and Active Status
sample_data = [
    (1, "alice smith", "HR", 55000, True),
    (2, "bob jones", "IT", 85000, True),
    (3, "charlie brown", "IT", 90000, False),
    (4, "david lee", "Marketing", 60000, True),
    (5, "eva green", "HR", 52000, True),
    (6, "frank white", "Marketing", 75000, False)
]

# Define the schema
schema = ["id", "name", "department", "salary", "is_active"]

# Create the DataFrame
df = spark.createDataFrame(sample_data, schema=schema)

print("--- Original Data ---")
df.show()

# 3. Apply Transformations
# Note: PySpark DataFrames are immutable, so we chain transformations or assign them to a new variable.
transformed_df = df \
    .filter(col("is_active") == True) \
    .withColumn("name", upper(col("name"))) \
    .withColumn("salary_bonus", col("salary") * 1.10) \
    .withColumn("tier", when(col("salary") >= 80000, "Senior").otherwise("Junior")) \
    .select("id", "name", "department", "salary_bonus", "tier")

print("--- Transformed Data (Filtered, Modified Columns, and Column Chaining) ---")
transformed_df.show()

# 4. Apply an Aggregation Transformation (GroupBy)
summary_df = transformed_df \
    .groupBy("department") \
    .agg(
        round(avg("salary_bonus"), 2).alias("average_bonus_salary")
    )

print("--- Aggregated Data (Average Salary by Department) ---")
summary_df.show()

# Stop the Spark Session
spark.stop()
